# CNN Product Image Classifier
**CPSC 393 – Fall 2026** · Luke Wiljanen, Ashlee Nguyen & Auggie Pirazzi

We compare four models on Kaggle's [Fashion Product Images (Small)](https://www.kaggle.com/datasets/paramaggarwal/fashion-product-images-small) to see where classification performance comes from: a fully connected network, a CNN from scratch, a convolutional autoencoder, and a fine-tuned ResNet50. All four use the same split, callbacks, and evaluation code, so the model is the only thing that changes.

| Section | Owner | Status |
|---|---|---|
| 1. Setup | All | ✅ |
| 2. Load and check the data | Auggie / Luke | ✅ |
| 3. Train / validation / test splits | Luke | ✅ |
| 4. Preprocessing and shared functions | Ashlee / Luke | ✅ |
| 5. Fully connected baseline | Luke | ✅ |
| 6. CNN from scratch | Ashlee | ⬜ |
| 7. ResNet50 (transfer learning) | Auggie | ⬜ |
| 8. Convolutional autoencoder | Luke | ✅ |
| 9. Final test evaluation and figures | All | ⬜ |

**Working in one notebook as a team:** only edit your own section, and run *Kernel → Restart & Clear Output* before committing. Notebook outputs cause most merge conflicts.

## 1. Setup

In [ ]:
# loading libraries for data manipulation
import numpy as np
import pandas as pd

# loading libraries for data visualization
import matplotlib.pyplot as plt
from plotnine import *
from PIL import Image

# import tensorflow and keras packages
import tensorflow as tf
from tensorflow import keras

# evaluation metrics and t-SNE
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.manifold import TSNE

# import kagglehub library to fetch data
import kagglehub

# import os, glob, and csv for file directory operations
import os, glob, csv

import warnings
warnings.filterwarnings('ignore')

print("TensorFlow", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

Settings shared by every model. Change them here only, so all four models stay comparable.

In [ ]:
SEED = 42
keras.utils.set_random_seed(SEED)

# folders
SPLITS_DIR = "splits"     # committed, so everyone uses the same split
FIGURES_DIR = "figures"
MODELS_DIR = "models"     # trained models (git-ignored)
for folder in [SPLITS_DIR, FIGURES_DIR, MODELS_DIR]:
    os.makedirs(folder, exist_ok=True)

# articleType values exactly as they appear in styles.csv
CLASSES_7 = ["Tshirts", "Casual Shoes", "Watches", "Kurtas", "Handbags", "Heels", "Sunglasses"]
CLASSES_EXTRA = ["Shirts", "Tops", "Sports Shoes"]   # added for the ten-class run
CLASSES_10 = CLASSES_7 + CLASSES_EXTRA

PER_CLASS = 1000
SPLIT_SIZES = {"train": 700, "val": 150, "test": 150}   # 70 / 15 / 15

# images are 60 pixels wide and 80 tall, so arrays have shape (80, 60, 3)
IMG_H, IMG_W = 80, 60
IMG_SHAPE = (IMG_H, IMG_W, 3)

# training settings from the proposal
BATCH_SIZE = 64
EPOCHS = 50            # early stopping usually ends training well before this
LR_SCRATCH = 1e-3      # models trained from scratch
LR_FINE_TUNE = 1e-4    # ResNet50
L2 = 1e-4              # weight decay

## 2. Load and check the data
We use the kaggle library to download the dataset (about 600 MB, cached after the first run).

In [ ]:
# pass dataset name as parameter
path = kagglehub.dataset_download("paramaggarwal/fashion-product-images-small")
print("Path to dataset files:", path)

# find styles.csv and the images folder next to it
styles_path = glob.glob(f"{path}/**/styles.csv", recursive=True)[0]
images_dir = os.path.join(os.path.dirname(styles_path), "images")
print(styles_path)
print(images_dir)

A few product names contain commas, so those rows have more than ten fields and `pd.read_csv` fails on them. We join the extra fields back into the product name so those products are kept instead of skipped.

In [ ]:
columns = ["id", "gender", "masterCategory", "subCategory", "articleType",
           "baseColour", "season", "year", "usage", "productDisplayName"]

rows = []
n_fixed = 0
with open(styles_path, newline="", encoding="utf-8") as f:
    reader = csv.reader(f)
    next(reader)  # skip header
    for r in reader:
        if len(r) > len(columns):  # extra commas in the product name
            r = r[:9] + [",".join(r[9:])]
            n_fixed += 1
        rows.append(r)

df = pd.DataFrame(rows, columns=columns).replace("", np.nan)
df["id"] = df["id"].astype(int)

# flag products that have an image file
image_ids = {int(os.path.splitext(f)[0]) for f in os.listdir(images_dir) if f.endswith(".jpg")}
df["has_image"] = df["id"].isin(image_ids)

print(f"Rows: {len(df):,}  (rows with commas in the name: {n_fixed})")
print(f"Article types: {df['articleType'].nunique()}")
print(f"Products with no image file: {(~df['has_image']).sum()}")
df.head()

In [ ]:
# missing values per column
df.isna().sum()

Let's look at how many products each of our classes has.

In [ ]:
have = df[df["has_image"]]

class_counts = (have[have["articleType"].isin(CLASSES_10)]["articleType"]
                .value_counts().rename_axis("articleType").reset_index(name="count"))
class_counts["run"] = np.where(class_counts["articleType"].isin(CLASSES_7), "seven-class", "ten-class only")

(
    ggplot(class_counts, aes(x="reorder(articleType, count)", y="count", fill="run"))
    + geom_col()
    + geom_hline(yintercept=PER_CLASS, linetype="dashed")
    + coord_flip()
    + labs(title="Products per class (dashed line = 1,000 cap)", x="", y="Products with an image", fill="")
    + theme_minimal()
)

We can also check the dimensions and color mode of every image.

In [ ]:
sizes = []
for pid in have["id"]:
    img = Image.open(os.path.join(images_dir, f"{pid}.jpg"))
    sizes.append((img.size, img.mode))  # (width, height), 'RGB' or 'L'

size_counts = pd.Series(sizes).value_counts()
print(size_counts)
print(f"\nImages that are not {IMG_W}x{IMG_H} RGB: {(pd.Series(sizes) != ((IMG_W, IMG_H), 'RGB')).sum()}")

The function below loads images as arrays. It converts grayscale images to RGB and resizes any image that is not 60×80, so every array has the same shape.

In [ ]:
def load_images(ids):
    images = np.zeros((len(ids), IMG_H, IMG_W, 3), dtype=np.uint8)
    for i, pid in enumerate(ids):
        img = Image.open(os.path.join(images_dir, f"{pid}.jpg")).convert("RGB")
        if img.size != (IMG_W, IMG_H):
            img = img.resize((IMG_W, IMG_H))
        images[i] = np.array(img)
    return images

In [ ]:
# display 8 examples from each class
rng = np.random.default_rng(SEED)
n = 8

plt.figure(figsize=(n * 1.1, len(CLASSES_7) * 1.45))
for r, label in enumerate(CLASSES_7):
    ids = rng.choice(have.loc[have["articleType"] == label, "id"].to_numpy(), size=n, replace=False)
    for j, img in enumerate(load_images(ids)):
        ax = plt.subplot(len(CLASSES_7), n, r * n + j + 1)
        plt.imshow(img)
        plt.xticks([]); plt.yticks([])
        if j == 0:
            plt.ylabel(label, rotation=0, ha="right", va="center", fontsize=9)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/sample_grid.png", dpi=150)
plt.show()

## 3. Train / validation / test splits
The splits are built once with a fixed seed and saved to `splits/`. If the files already exist, we load them instead of re-splitting, so every model and every teammate uses the same images.

- **Seven-class:** 1,000 images per class → 4,900 train / 1,050 validation / 1,050 test.
- **Ten-class:** the same seven-class split, plus 1,000 each of shirts, tops, and sports shoes.
- **Unlabeled pool** (autoencoder): every product with an image except the test images of *both* runs, so no test image influences feature learning.

In [ ]:
split_files = {"seven": f"{SPLITS_DIR}/split_7class.csv",
               "ten": f"{SPLITS_DIR}/split_10class.csv",
               "pool": f"{SPLITS_DIR}/unlabeled_pool.csv"}

# sample 1,000 ids from one class and divide them 700 / 150 / 150
def split_class(label, rng):
    ids = np.sort(have.loc[have["articleType"] == label, "id"].to_numpy())
    ids = rng.permutation(ids)[:PER_CLASS]
    parts = []
    start = 0
    for split, size in SPLIT_SIZES.items():
        parts.append(pd.DataFrame({"id": ids[start:start + size], "label": label, "split": split}))
        start += size
    return pd.concat(parts)

if all(os.path.exists(f) for f in split_files.values()):
    split_7 = pd.read_csv(split_files["seven"])
    split_10 = pd.read_csv(split_files["ten"])
    pool = pd.read_csv(split_files["pool"])
    print("Loaded existing splits from", SPLITS_DIR)
else:
    rng = np.random.default_rng(SEED)
    split_7 = pd.concat([split_class(c, rng) for c in CLASSES_7], ignore_index=True)
    extra = pd.concat([split_class(c, rng) for c in CLASSES_EXTRA], ignore_index=True)
    split_10 = pd.concat([split_7, extra], ignore_index=True)

    # every product with an image, minus the test images of both runs
    test_ids = set(split_10.loc[split_10["split"] == "test", "id"])
    pool = have.loc[~have["id"].isin(test_ids), ["id"]].sort_values("id").reset_index(drop=True)

    split_7.to_csv(split_files["seven"], index=False)
    split_10.to_csv(split_files["ten"], index=False)
    pool.to_csv(split_files["pool"], index=False)
    print("Built and saved new splits to", SPLITS_DIR, "- commit these files.")

In [ ]:
# sanity checks: balanced classes, no image in two splits, no test image in the pool
for name, split_df in [("seven-class", split_7), ("ten-class", split_10)]:
    assert split_df["id"].is_unique, f"{name}: an image appears in two splits"
    print(name)
    print(pd.crosstab(split_df["label"], split_df["split"])[list(SPLIT_SIZES)], "\n")

test_ids = set(split_10.loc[split_10["split"] == "test", "id"])
assert not test_ids & set(pool["id"]), "a test image leaked into the unlabeled pool"
print(f"Unlabeled pool: {len(pool):,} images")

## 4. Preprocessing and shared functions
Every model uses the arrays, augmentation, callbacks, and evaluation functions in this section.

Let's load the seven-class images, scale pixel values to [0, 1], and one-hot encode the labels.

In [ ]:
def prepare_split(split_df, class_names):
    data = {}
    for split in SPLIT_SIZES:
        part = split_df[split_df["split"] == split]
        X = load_images(part["id"].to_numpy()).astype("float32") / 255.0
        y = part["label"].map({c: i for i, c in enumerate(class_names)}).to_numpy()
        data[split] = (X, keras.utils.to_categorical(y, num_classes=len(class_names)))
    return data

data_7 = prepare_split(split_7, CLASSES_7)
X_train, y_train = data_7["train"]
X_val, y_val = data_7["val"]
X_test, y_test = data_7["test"]

print("Train:", X_train.shape, "Validation:", X_val.shape, "Test:", X_test.shape)

The models trained from scratch standardize each color channel using the mean and standard deviation of the **training** split. A `Normalization` layer learns those statistics once with `adapt()` and then applies them inside the model.

ResNet50 does not use this layer. It uses its own `preprocess_input` instead.

In [ ]:
normalizer = keras.layers.Normalization(axis=-1)
normalizer.adapt(X_train)  # learns the channel mean and variance from the training images only

print("Channel means:", normalizer.mean.numpy().ravel())

Data augmentation adds variation to the training images: horizontal flips, small rotations, and mild color changes. No vertical flips, since upside-down products don't occur in a catalog. Keras only applies these layers during training, so validation and test images are never changed.

In [ ]:
# Create a new layer of data augmentation
data_augmentation = keras.Sequential([
        keras.layers.RandomFlip("horizontal"),
        keras.layers.RandomRotation(0.05),
        keras.layers.RandomBrightness(0.1, value_range=(0, 1)),
        keras.layers.RandomContrast(0.1)
], name="data_augmentation")

# Test augmentation on a single input image
sample_img = tf.expand_dims(X_train[0], 0)

plt.figure(figsize=(10, 4))
for i in range(8):
    aug_img = data_augmentation(sample_img, training=True)
    plt.subplot(2, 4, i + 1)
    plt.imshow(np.clip(aug_img[0], 0, 1))
    plt.axis('off')
plt.suptitle("Augmented versions of X_train[0]")
plt.tight_layout()
plt.show()

Early stopping halts training when validation loss stops improving. ReduceLROnPlateau lowers the learning rate when progress stalls.

In [ ]:
def make_callbacks():
    early_stopping = keras.callbacks.EarlyStopping(
        monitor = "val_loss", # track validation loss
        patience = 6, # wait 6 epochs for improvement
        restore_best_weights = True # use best weights learned as final set
    )
    reduce_lr = keras.callbacks.ReduceLROnPlateau(
        monitor = "val_loss",
        factor = 0.5, # halve the learning rate
        patience = 3, # after 3 epochs without improvement
        min_lr = 1e-6
    )
    return [early_stopping, reduce_lr]

This function takes a model's history and plots the training and validation loss and accuracy per epoch.

In [ ]:
# function will take a NN history and plot the training/validation loss and accuracy per epoch
def evaluation_plots(model_history, title=""):
    # Extract metrics from the history object
    history_dict = model_history.history
    loss = history_dict['loss']
    val_loss = history_dict['val_loss']
    accuracy = history_dict['accuracy']
    val_accuracy = history_dict['val_accuracy']

    epochs = range(1, len(loss) + 1)

    # Create a figure with 2 subplots (side-by-side)
    plt.figure(figsize=(14, 5))

    # Plot 1: Loss Curves
    plt.subplot(1, 2, 1)
    plt.plot(epochs, loss, 'r--', label='Training Loss', linewidth=2)
    plt.plot(epochs, val_loss, 'b-', label='Validation Loss', linewidth=2)
    plt.title(f'Training and Validation Loss ({title})', fontsize=12)
    plt.xlabel('Epochs', fontsize=10)
    plt.ylabel('Loss', fontsize=10)
    plt.legend(fontsize=10)
    plt.grid(True)

    # Plot 2: Accuracy Curves
    plt.subplot(1, 2, 2)
    plt.plot(epochs, accuracy, 'r--', label='Training Accuracy', linewidth=2)
    plt.plot(epochs, val_accuracy, 'b-', label='Validation Accuracy', linewidth=2)
    plt.title(f'Training and Validation Accuracy ({title})', fontsize=12)
    plt.xlabel('Epochs', fontsize=10)
    plt.ylabel('Accuracy', fontsize=10)
    plt.legend(fontsize=10)
    plt.grid(True)

    plt.tight_layout()
    plt.show()

This function reports accuracy and macro F1, plots the confusion matrix, and computes our business measure: the share of images the model is at least 90% confident about, and its accuracy on that subset. Use the validation set while tuning. The test set is used once, in section 9.

In [ ]:
def evaluate_model(model, X, y, class_names, name, confidence=0.90, plot=True):
    # predictions are provided as probabilities per class, we pick the highest one
    y_pred_probs = model.predict(X, verbose=0)
    y_pred = np.argmax(y_pred_probs, axis=1)  # predicted classes
    y_true = np.argmax(y, axis=1)             # actual classes

    # share of images the model could tag automatically, and its accuracy on them
    confident = y_pred_probs.max(axis=1) >= confidence

    results = {
        "model": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        f"auto_tag_share@{confidence:.0%}": confident.mean(),
        f"auto_tag_accuracy@{confidence:.0%}": (y_pred[confident] == y_true[confident]).mean() if confident.any() else np.nan,
    }

    if plot:
        cm = confusion_matrix(y_true, y_pred)
        fig, ax = plt.subplots(figsize=(7, 6))
        ConfusionMatrixDisplay(cm, display_labels=class_names).plot(ax=ax, cmap="Blues", xticks_rotation=45, colorbar=False)
        plt.title(f"Confusion Matrix ({name})")
        plt.tight_layout()
        plt.show()

    return results

We can also look at examples the model got right and wrong.

In [ ]:
# display up to n correct and incorrect predictions
def show_images(model, X, y, class_names, n=8):
    y_pred = np.argmax(model.predict(X, verbose=0), axis=1)  # predicted classes
    y_true = np.argmax(y, axis=1)                             # actual classes

    # separate correct and incorrect predictions
    correct_indices = np.where(y_pred == y_true)[0]
    incorrect_indices = np.where(y_pred != y_true)[0]

    for indices, title in [(correct_indices, "Correct!"), (incorrect_indices, "Incorrect!")]:
        plt.figure(figsize=(14, 3))
        for i, idx in enumerate(indices[:n]):
            plt.subplot(1, n, i + 1)
            plt.imshow(X[idx])
            plt.title(f"True: {class_names[y_true[idx]]}\nPred: {class_names[y_pred[idx]]}", fontsize=8)
            plt.axis('off')
        plt.suptitle(title)
        plt.tight_layout()
        plt.show()

In [ ]:
# collects validation results from every model for comparison
val_results = []

## 5. Fully connected baseline (Luke)
This network flattens each image into a vector of 80 × 60 × 3 = 14,400 values, so it has no sense of where pixels sit relative to each other. It has two hidden layers (512 and 128 units) with L2 weight decay and dropout.

`model.summary()` should show **7,439,879 trainable parameters**. The few non-trainable ones are the normalization layer's mean and variance.

In [ ]:
def build_fc_model(num_classes):
    model = keras.Sequential([
        keras.layers.Input(shape=IMG_SHAPE),
        data_augmentation, # only active during training
        normalizer, # standardize each color channel
        keras.layers.Flatten(), # 80x60x3 image -> vector of 14,400 values
        keras.layers.Dense(512, activation='relu', kernel_regularizer=keras.regularizers.l2(L2)), # hidden layer 1
        keras.layers.Dropout(0.5),
        keras.layers.Dense(128, activation='relu', kernel_regularizer=keras.regularizers.l2(L2)), # hidden layer 2
        keras.layers.Dropout(0.5),
        keras.layers.Dense(num_classes, activation='softmax') # output layer
    ], name="fully_connected")
    return model

fc_model = build_fc_model(len(CLASSES_7))
fc_model.summary()

In [ ]:
fc_model.compile(loss="categorical_crossentropy",
                 optimizer=keras.optimizers.Adam(learning_rate=LR_SCRATCH),
                 metrics=["accuracy"])

history_fc = fc_model.fit(X_train, y_train,
                          epochs=EPOCHS, verbose=1, batch_size=BATCH_SIZE,
                          validation_data=(X_val, y_val),
                          callbacks=make_callbacks())

In [ ]:
evaluation_plots(history_fc, "Fully Connected")

We expect a visible gap between training and validation loss here. With about 1,518 parameters per training image, this model is the most likely to overfit, and that gap is one of our findings.

In [ ]:
val_results.append(evaluate_model(fc_model, X_val, y_val, CLASSES_7, "Fully Connected"))
show_images(fc_model, X_val, y_val, CLASSES_7)
pd.DataFrame(val_results)

In [ ]:
fc_model.save("models/fc_7class.keras")

## 6. CNN from scratch (Ashlee)
Three blocks of 3×3 convolution (32, 64, then 128 filters, stride 1, "same" padding), each followed by batch normalization, ReLU, and 2×2 max pooling. Then global average pooling and the output layer.

- Use the **functional** API, as in Lab 07, so section 9 can pull out the conv layers for feature maps and Grad-CAM. Give the conv layers names that include `conv`.
- Put `data_augmentation` and `normalizer` right after the input, like `build_fc_model` above.
- Batch normalization goes between the convolution and the activation, as in Lab 05.
- Expect **94,599 trainable** parameters. The summary's total will be a little higher (about 95,054), because batch norm's moving averages and the normalization layer's statistics aren't trained.
- Compile, fit, and evaluate the same way as section 5: `make_callbacks()`, `evaluation_plots`, `evaluate_model`, `show_images`, and append to `val_results`.

In [ ]:
# TODO (Ashlee): build_cnn_model(num_classes), train it as cnn_model, save to models/cnn_7class.keras

## 7. ResNet50 with transfer learning (Auggie)
Follow the Lab 09 pattern: load the base model without its top layer, freeze it, add `GlobalAveragePooling2D` and a new output layer, train, then unfreeze and fine-tune with a lower learning rate.

- `keras.applications.ResNet50(include_top=False, weights="imagenet", input_shape=(128, 96, 3))`.
- Our arrays are scaled to [0, 1], so before the base model add `keras.layers.Resizing(128, 96)` and `keras.layers.Rescaling(255.0)`, then `keras.applications.resnet50.preprocess_input`. **Don't** use `normalizer`.
- `data_augmentation` can stay as the first layer.
- Fine-tune at `LR_FINE_TUNE` (1e-4). When you unfreeze, recompile the same model you're training. Lab 09 recompiles `model` instead of `tl_model`, so copy carefully.
- Evaluate the same way as section 5 and append to `val_results`.

In [ ]:
# TODO (Auggie): build and train resnet_model, save to models/resnet50_7class.keras

## 8. Convolutional autoencoder (Luke)
The autoencoder learns features from the whole unlabeled pool, with no labels. Three stride-2 convolutions reduce each 80×60×3 image to 10×8×128 = 10,240 values, which a dense layer compresses into a 128-number latent code. The decoder mirrors this back to an image.

It trains on reconstruction error (MSE), using the input as both the input and the target, as in Lab 08.

In [ ]:
# load the unlabeled pool as 8-bit images to save memory (about 620 MB)
X_pool = load_images(pool["id"].to_numpy())

# hold out 5% of the pool to monitor reconstruction loss
rng = np.random.default_rng(SEED)
pool_order = rng.permutation(len(X_pool))
n_hold = int(0.05 * len(X_pool))
pool_val_idx, pool_train_idx = pool_order[:n_hold], pool_order[n_hold:]

# input and output are the same image, scaled to [0, 1]
def to_pair(x):
    x = tf.cast(x, tf.float32) / 255.0
    return x, x

pool_train = (tf.data.Dataset.from_tensor_slices(X_pool[pool_train_idx])
              .shuffle(5000, seed=SEED).batch(BATCH_SIZE).map(to_pair).prefetch(tf.data.AUTOTUNE))
pool_val = (tf.data.Dataset.from_tensor_slices(X_pool[pool_val_idx])
            .batch(BATCH_SIZE).map(to_pair).prefetch(tf.data.AUTOTUNE))

print(f"Autoencoder training images: {len(pool_train_idx):,}, held out: {n_hold:,}")

In [ ]:
LATENT_DIM = 128

# define input dimensions
input_img = keras.layers.Input(shape=IMG_SHAPE)

# encoder
x = keras.layers.Conv2D(32, (3,3), strides=2, activation="relu", padding="same")(input_img) # (40,30,32)
x = keras.layers.Conv2D(64, (3,3), strides=2, activation="relu", padding="same")(x) # (20,15,64)
x = keras.layers.Conv2D(128, (3,3), strides=2, activation="relu", padding="same")(x) # (10,8,128)
x = keras.layers.Flatten()(x) # 10,240 values
encoded = keras.layers.Dense(LATENT_DIM, activation="relu", name="latent")(x) # latent code (128)

# decoder
x = keras.layers.Dense(10 * 8 * 128, activation="relu")(encoded)
x = keras.layers.Reshape((10, 8, 128))(x) # (10,8,128)
x = keras.layers.Conv2DTranspose(128, (3,3), strides=2, activation="relu", padding="same")(x) # (20,16,128)
x = keras.layers.Conv2DTranspose(64, (3,3), strides=2, activation="relu", padding="same")(x) # (40,32,64)
x = keras.layers.Conv2DTranspose(32, (3,3), strides=2, activation="relu", padding="same")(x) # (80,64,32)
x = keras.layers.Cropping2D(((0, 0), (2, 2)))(x) # (80,60,32), trims the 2 extra columns on each side
decoded = keras.layers.Conv2D(3, (3,3), activation="sigmoid", padding="same")(x) # (80,60,3)

autoencoder = keras.Model(input_img, decoded, name="autoencoder")
autoencoder.summary()

In [ ]:
# compile model
autoencoder.compile(optimizer=keras.optimizers.Adam(LR_SCRATCH), loss="mse")

# train model
history_ae = autoencoder.fit(
    pool_train,
    epochs=EPOCHS,
    validation_data=pool_val,
    callbacks=make_callbacks()[:1],  # early stopping only
    verbose=1
)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history_ae.history['loss'], 'r--', label='Training Loss', linewidth=2)
plt.plot(history_ae.history['val_loss'], 'b-', label='Validation Loss', linewidth=2)
plt.title('Autoencoder Reconstruction Loss (MSE)')
plt.xlabel('Epochs'); plt.ylabel('MSE')
plt.legend(); plt.grid(True)
plt.show()

**Nov 20 checkpoint:** if the reconstructions below are recognizable (a shoe still reads as a shoe, a watch as a watch), we keep the autoencoder. If not, we drop it and report the three supervised models.

In [ ]:
# predict using network
X_val_reconstructed = autoencoder.predict(X_val, verbose=0)

# visualization function
def plot_reconstructions(input, ae_output, n=10, title=""):
    plt.figure(figsize=(16, 4))
    for i in range(n):
        # Original
        ax = plt.subplot(2, n, i + 1)
        plt.imshow(input[i])
        plt.title("Original")
        plt.axis('off')

        # Reconstructed
        ax = plt.subplot(2, n, i + 1 + n)
        plt.imshow(ae_output[i])
        plt.title("Reconstructed")
        plt.axis('off')
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

# show 10 validation images spread across the classes
sample = np.arange(0, len(X_val), len(X_val) // 10)[:10]
plot_reconstructions(X_val[sample], X_val_reconstructed[sample])

Now we freeze the encoder and train a small classifier on its 128-number latent codes, using the same labeled split as the other models.

In [ ]:
# first we extract the trained encoder from the network
encoder = keras.Model(inputs=autoencoder.input, outputs=encoded, name="encoder")
encoder.trainable = False

# next we get the latent representation of the labeled images
Z_train = encoder.predict(X_train, verbose=0)
Z_val = encoder.predict(X_val, verbose=0)
print("Latent codes:", Z_train.shape)

In [ ]:
ae_classifier = keras.Sequential([
    keras.layers.Input(shape=(LATENT_DIM,)),
    keras.layers.BatchNormalization(), # latent codes are not standardized
    keras.layers.Dense(64, activation='relu', kernel_regularizer=keras.regularizers.l2(L2)),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(len(CLASSES_7), activation='softmax') # output layer
], name="ae_classifier")

ae_classifier.compile(loss="categorical_crossentropy",
                      optimizer=keras.optimizers.Adam(learning_rate=LR_SCRATCH),
                      metrics=["accuracy"])

history_ae_clf = ae_classifier.fit(Z_train, y_train,
                                   epochs=EPOCHS, verbose=1, batch_size=BATCH_SIZE,
                                   validation_data=(Z_val, y_val),
                                   callbacks=make_callbacks())
evaluation_plots(history_ae_clf, "Autoencoder Classifier")

In [ ]:
# combine encoder + classifier so it takes images like the other models
ae_model = keras.Sequential([keras.layers.Input(shape=IMG_SHAPE), encoder, ae_classifier], name="autoencoder_classifier")

val_results.append(evaluate_model(ae_model, X_val, y_val, CLASSES_7, "Autoencoder"))
pd.DataFrame(val_results)

Let's use t-SNE to plot the latent codes in two dimensions, colored by class. Classes that form separate clusters are ones the encoder learned to tell apart without labels.

In [ ]:
# now we run t-SNE on these vectors
tsne = TSNE(n_components=2, perplexity=30, random_state=SEED)
latent_2d = tsne.fit_transform(Z_val)

# plot the representation in 2 dimensions
val_labels = np.argmax(y_val, axis=1)
plt.figure(figsize=(10, 8))
for i, label in enumerate(CLASSES_7):
    pts = latent_2d[val_labels == i]
    plt.scatter(pts[:, 0], pts[:, 1], s=10, label=label, color=plt.cm.tab10(i))
plt.title(f"t-SNE of Encoded Products ({LATENT_DIM}D → 2D)")
plt.xlabel("t-SNE Dimension 1")
plt.ylabel("t-SNE Dimension 2")
plt.legend(markerscale=2)
plt.grid(True)
plt.savefig(f"{FIGURES_DIR}/ae_tsne.png", dpi=150)
plt.show()

In [ ]:
autoencoder.save("models/autoencoder.keras")
ae_classifier.save("models/ae_classifier_7class.keras")

## 9. Final test evaluation and figures (All)
**Run this section once,** after all tuning is finished. The test set is never used before this point.

In [ ]:
# TODO (All): uncomment the models that are finished
final_models = {
    "Fully Connected": fc_model,
    # "CNN": cnn_model,
    "Autoencoder": ae_model,
    # "ResNet50": resnet_model,
}

test_results = [evaluate_model(m, X_test, y_test, CLASSES_7, name) for name, m in final_models.items()]
test_table = pd.DataFrame(test_results).round(3)
test_table.to_csv(f"{FIGURES_DIR}/test_results_7class.csv", index=False)
test_table

Feature maps show what each convolution layer responds to (Lab 07). They need the functional CNN from section 6.

In [ ]:
# Display feature maps (the activations produced by convolutional layers)
def plot_feature_maps(model, img, max_channels=10):
    # Create model for intermediate activations
    conv_layers = [layer for layer in model.layers if 'conv' in layer.name]
    activation_model = keras.models.Model(inputs=model.input, outputs=[layer.output for layer in conv_layers])

    # Get activations by "predicting" for each conv layer
    activations = activation_model.predict(img[np.newaxis], verbose=0)
    if not isinstance(activations, list):
        activations = [activations]

    for layer, activation in zip(conv_layers, activations):
        display_channels = min(max_channels, activation.shape[-1])

        fig, axes = plt.subplots(1, display_channels + 1, figsize=(15, 3))
        fig.suptitle(f'Feature maps from layer: {layer.name}', fontsize=12)
        axes[0].imshow(img)
        axes[0].set_title("Input")
        for i in range(display_channels):
            axes[i + 1].matshow(activation[0, :, :, i], cmap='viridis')
        for ax in axes:
            ax.axis('off')
        plt.show()

# TODO (Ashlee): plot_feature_maps(cnn_model, X_test[0])

Grad-CAM shows which parts of an image drove the prediction (Lab 07). Use it on a few correct and incorrect predictions.

In [ ]:
import cv2

def grad_cam(model, img, last_conv_layer_name=None):
    # Step 1. make a prediction
    img_batch = img[np.newaxis]
    pred_class = np.argmax(model.predict(img_batch, verbose=0)[0])

    # Step 2. compute gradient of the class score with respect to the last convolution layer
    if last_conv_layer_name is None:
        last_conv_layer_name = [layer.name for layer in model.layers if 'conv' in layer.name][-1]
    grad_model = keras.models.Model(model.inputs, [model.get_layer(last_conv_layer_name).output, model.output])

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_batch)
        loss = predictions[:, pred_class]
    grads = tape.gradient(loss, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2)) # global average pooling over gradients

    # Step 3. multiply each channel in the conv output by its gradient importance
    heatmap = conv_outputs[0] @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = (tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-8)).numpy() # normalize to [0, 1]
    heatmap_resized = cv2.resize(heatmap, (IMG_W, IMG_H))

    # Step 4. plot heatmap
    image_rgb = np.uint8(255 * img)
    heatmap_color = cv2.cvtColor(cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET), cv2.COLOR_BGR2RGB)
    superimposed_img = cv2.addWeighted(image_rgb, 0.6, heatmap_color, 0.4, 0)

    plt.figure(figsize=(9, 4))
    for i, (title, im) in enumerate([("Original Image", img), ("Grad-CAM Heatmap", heatmap_resized), ("Grad-CAM Overlay", superimposed_img)]):
        plt.subplot(1, 3, i + 1)
        plt.title(title)
        plt.imshow(im, cmap='jet' if i == 1 else None)
        plt.axis('off')
    plt.tight_layout()
    plt.show()
    return pred_class

# TODO (Auggie): grad_cam(cnn_model, X_test[0]); for ResNet50 pass its last conv layer name

These are the test images the autoencoder reconstructs worst, which shows what kinds of products it never learned to represent.

In [ ]:
X_test_reconstructed = autoencoder.predict(X_test, verbose=0)
errors = np.mean((X_test - X_test_reconstructed) ** 2, axis=(1, 2, 3))
worst = np.argsort(errors)[::-1][:10]

plot_reconstructions(X_test[worst], X_test_reconstructed[worst], title="Highest reconstruction error (test set)")

### Ten-class run
Repeat sections 5–8 with `data_10 = prepare_split(split_10, CLASSES_10)` and `len(CLASSES_10)` outputs, using the same settings. Re-adapt `normalizer` on the ten-class training images first. The autoencoder doesn't need retraining, since its pool already excludes the ten-class test images. Only the small classifier is retrained.

In [ ]:
# TODO (All)